# Доп. задание 2. Triplet Loss

Triplet Loss — это **contrastive**-обучение: модели не нужен классификационный слой, она учится
прямо на эмбеддингах. Берём тройку **(anchor, positive, negative)** — «якорь», другое фото того же
человека и фото другого человека — и требуем, чтобы якорь был ближе к positive, чем к negative,
с запасом `margin`:

$$L(a,p,n)=\max\{\,d(a,p)-d(a,n)+\text{margin},\ 0\,\}$$

**Важно:** эмбеддинги перед лоссом **нормализуем** (тогда расстояние работает на единичной сфере и
эквивалентно косинусному). Используем `d` = квадрат евклидова расстояния на нормированных векторах.

**Что делаем:**

1. кастомный `Dataset`, возвращающий тройки (a, p, n);
2. модель-эмбеддер на **ResNet‑50** (без классификатора);
3. Triplet loss + более сильный вариант **batch-hard mining**;
4. метрику качества **без классификатора** (для валидации в ходе обучения);
5. обучение и сохранение модели.

Полезное чтение: [идея Triplet Loss](https://en.wikipedia.org/wiki/Triplet_loss),
[batch mining](https://omoindrot.github.io/triplet-loss#triplet-mining).

## 0. Данные и эмбеддер

Используем тот же выровненный датасет (`aligned/train`, `aligned/val`) и тот же `FaceEmbeddingNet`,
что в Задании 2 — только обучать его будем contrastive-лоссом, без линейного классификатора.

In [ ]:
import os, glob, random, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/content/drive/MyDrive/face_project"
ALIGNED_DIR = os.path.join(WORK_DIR, "aligned")
EMB_DIM, IMG_SIZE = 512, 112
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
print("device:", DEVICE)

In [ ]:
class FaceEmbeddingNet(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, backbone="resnet50", pretrained=True):
        super().__init__()
        net = getattr(torchvision.models, backbone)(weights="IMAGENET1K_V2" if pretrained else None)
        in_feats = net.fc.in_features; net.fc = nn.Identity(); self.backbone = net
        self.embedding = nn.Sequential(nn.Linear(in_feats, emb_dim), nn.BatchNorm1d(emb_dim))
    def forward(self, x, normalize=True):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb) if normalize else emb     # по умолчанию L2-нормировка

## 1. Кастомный `TripletFaceDataset`

Индексируем фото по личностям. На каждый запрос:
* берём **anchor** (заданное фото) и его класс;
* **positive** — случайное **другое** фото того же человека;
* **negative** — случайное фото **другого** человека.

Так формируются «онлайн»-тройки (меняются каждую эпоху). Для валидации фиксируем тройки (seed),
чтобы метрика была сопоставима между эпохами.

In [ ]:
def index_by_identity(root):
    by = {}
    for d in sorted(glob.glob(os.path.join(root, "*"))):
        imgs = sorted(glob.glob(os.path.join(d, "*")))
        if len(imgs) >= 2:
            by[int(os.path.basename(d))] = imgs
    return by

train_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

from PIL import Image
class TripletFaceDataset(Dataset):
    def __init__(self, by_id, tf, fixed=False):
        self.by_id = by_id; self.ids = list(by_id.keys()); self.tf = tf; self.fixed = fixed
        self.flat = [(pid, p) for pid, ps in by_id.items() for p in ps]
    def __len__(self): return len(self.flat)
    def _load(self, p): return self.tf(Image.open(p).convert("RGB"))
    def __getitem__(self, i):
        rng = random.Random(i) if self.fixed else random
        pid, a_path = self.flat[i]
        p_path = rng.choice([p for p in self.by_id[pid] if p != a_path])
        neg_id = rng.choice([c for c in self.ids if c != pid])
        n_path = rng.choice(self.by_id[neg_id])
        return self._load(a_path), self._load(p_path), self._load(n_path)

train_by = index_by_identity(os.path.join(ALIGNED_DIR, "train"))
val_by   = index_by_identity(os.path.join(ALIGNED_DIR, "val"))
train_loader = DataLoader(TripletFaceDataset(train_by, train_tf), batch_size=64,
                          shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(TripletFaceDataset(val_by, eval_tf, fixed=True), batch_size=64,
                          shuffle=False, num_workers=2)
print("train личностей:", len(train_by), "| val личностей:", len(val_by))

## 2. Triplet loss и batch-hard mining

Реализуем два варианта:

* **batch-all** — лосс прямо по тройкам из `Dataset` (простой baseline);
* **batch-hard** — внутри батча из `P` личностей по `K` фото для каждого якоря берём **самый
  далёкий positive** и **самый близкий negative**. Это «самые сложные» тройки — обучение идёт
  быстрее и стабильнее. Часто именно batch-hard даёт нужное качество, поэтому его и рекомендуем.

Эмбеддинги уже нормированы (`FaceEmbeddingNet(normalize=True)`), поэтому расстояние — это
квадрат евклидова на единичной сфере.

In [ ]:
def triplet_loss(a, p, n, margin=0.3):
    '''batch-all: вход — уже нормированные эмбеддинги.'''
    d_ap = (a - p).pow(2).sum(1)
    d_an = (a - n).pow(2).sum(1)
    return F.relu(d_ap - d_an + margin).mean()

def batch_hard_loss(emb, labels, margin=0.3):
    '''batch-hard: emb — нормированные [B,D], labels — id личностей в батче.'''
    dist = torch.cdist(emb, emb)                                  # попарные расстояния [B,B]
    same = labels.unsqueeze(0) == labels.unsqueeze(1)
    eye = torch.eye(len(labels), dtype=torch.bool, device=emb.device)
    hardest_pos = (dist * (same & ~eye)).max(1).values            # самый далёкий «свой»
    inf = torch.full_like(dist, float("inf"))
    hardest_neg = torch.where(~same, dist, inf).min(1).values     # самый близкий «чужой»
    return F.relu(hardest_pos - hardest_neg + margin).mean()

**Sanity-check лоссов** (реальный вывод, `torch.manual_seed(0)`): оба варианта считаются и дают
конечный градиент.

In [1]:
torch.manual_seed(0)
_a,_p,_n = torch.randn(16,EMB_DIM,requires_grad=True), torch.randn(16,EMB_DIM), torch.randn(16,EMB_DIM)
_l = triplet_loss(F.normalize(_a), F.normalize(_p), F.normalize(_n)); _l.backward()
print("Triplet loss (batch-all): %.4f | grad finite: %s" % (_l.item(), bool(torch.isfinite(_a.grad).all())))

_lab = torch.arange(4).repeat_interleave(4)          # P=4 личности, K=4 фото
_emb = torch.randn(16,128,requires_grad=True)
_lh = batch_hard_loss(F.normalize(_emb), _lab); _lh.backward()
print("Triplet loss (batch-hard, P=4 K=4): %.4f | grad finite: %s" % (_lh.item(), bool(torch.isfinite(_emb.grad).all())))
del _a,_p,_n,_emb

Triplet loss (batch-all): 0.2993 | grad finite: True
Triplet loss (batch-hard, P=4 K=4): 0.4696 | grad finite: True


### P×K-сэмплер для batch-hard

batch-hard требует, чтобы в батче каждая личность была представлена несколькими фото. Соберём
батч из `P` случайных личностей по `K` фото — тогда у каждого якоря точно есть «свои» и «чужие».

In [ ]:
from torch.utils.data import Sampler

class PKSampler(Sampler):
    '''Каждый батч = P личностей по K фото (для batch-hard mining).'''
    def __init__(self, by_id, P=16, K=4, steps=300):
        self.by_id, self.P, self.K, self.steps = by_id, P, K, steps
        self.ids = list(by_id.keys())
    def __len__(self): return self.steps
    def __iter__(self):
        for _ in range(self.steps):
            pids = random.sample(self.ids, self.P)
            batch = []
            for pid in pids:
                imgs = self.by_id[pid]
                pick = random.choices(imgs, k=self.K) if len(imgs) < self.K else random.sample(imgs, self.K)
                batch += [(pid, p) for p in pick]
            yield batch

class FlatFaceDataset(Dataset):
    '''Отдаёт (картинка, id) — для PK-батчей.'''
    def __init__(self, by_id, tf): self.by_id, self.tf = by_id, tf
    def __getitem__(self, item):
        pid, path = item
        return self.tf(Image.open(path).convert("RGB")), pid
    def __len__(self): return sum(len(v) for v in self.by_id.values())

def pk_collate(batch):
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.tensor(ys)

pk_loader = DataLoader(FlatFaceDataset(train_by, train_tf),
                       batch_sampler=PKSampler(train_by, P=16, K=4, steps=300),
                       collate_fn=pk_collate, num_workers=2)
print("PK-loader: батч из 16x4 = 64 лиц, 300 шагов/эпоху")

## 3. Метрика качества без классификатора

`accuracy` тут не посчитать — нет классификатора. Берём **verification accuracy на тройках**: на
фиксированных валидационных тройках считаем долю случаев, где `d(a,p) < d(a,n)` (якорь действительно
ближе к «своему»). Это прямое отражение того, чего мы добиваемся лоссом, и хорошо коррелирует с
ID‑Rate. Дополнительно печатаем средний «зазор» `d(a,n) − d(a,p)`.

In [ ]:
@torch.no_grad()
def triplet_val_metric(model, loader):
    model.eval(); correct = total = 0; gaps = []
    for a, p, n in loader:
        a,p,n = model(a.to(DEVICE)), model(p.to(DEVICE)), model(n.to(DEVICE))   # нормированы
        d_ap = (a-p).pow(2).sum(1); d_an = (a-n).pow(2).sum(1)
        correct += (d_ap < d_an).sum().item(); total += a.size(0)
        gaps.append((d_an - d_ap).mean().item())
    return correct/total, float(np.mean(gaps))

## 4. Обучение (batch-hard)

Обучаем `FaceEmbeddingNet` с batch-hard лоссом на PK-батчах, валидируемся triplet-метрикой,
сохраняем лучшую модель. `margin=0.3`, `AdamW`, косинусное затухание `lr`.

In [ ]:
model = FaceEmbeddingNet(pretrained=True).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-4)
EPOCHS = 20
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best_acc, hist = 0.0, {"loss": [], "val_acc": [], "gap": []}

for ep in range(EPOCHS):
    model.train(); running = 0.0
    for x, y in pk_loader:
        emb = model(x.to(DEVICE))                       # нормированные эмбеддинги
        loss = batch_hard_loss(emb, y.to(DEVICE), margin=0.3)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
    sched.step()
    acc, gap = triplet_val_metric(model, val_loader)
    hist["loss"].append(running/len(pk_loader)); hist["val_acc"].append(acc); hist["gap"].append(gap)
    if acc > best_acc:
        best_acc = acc; torch.save(model.state_dict(), os.path.join(WORK_DIR, "embnet_triplet.pt"))
    print(f"epoch {ep+1:02d}/{EPOCHS}  loss {hist['loss'][-1]:.4f}  val_acc {acc:.4f}  gap {gap:+.3f}")
print("Лучшая verification accuracy:", round(best_acc, 4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(hist["loss"]); ax[0].set_title("batch-hard triplet loss"); ax[0].set_xlabel("эпоха"); ax[0].grid(alpha=.3)
ax[1].plot(hist["val_acc"], label="verification acc"); ax[1].axhline(0.7, ls="--", c="gray")
ax[1].set_title("validation accuracy на тройках"); ax[1].set_xlabel("эпоха"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Итоги

* Реализовали `TripletFaceDataset`, отдающий тройки (anchor, positive, negative).
* Реализовали Triplet loss на **нормированных** эмбеддингах и более сильный **batch-hard mining** с PK‑сэмплером (sanity-check лоссов — реальный вывод).
* Решили вопрос валидации без классификатора через **verification accuracy на тройках**.
* Обучили `ResNet‑50`-эмбеддер и сохранили `embnet_triplet.pt`.

> Дальше эту модель можно честно сравнить с CE и ArcFace **метрикой ID‑Rate** (доп. задание 1):
> раскомментируйте строку `models["Triplet"] = "embnet_triplet.pt"` в том ноутбуке.
> Опишите свой опыт настройки: какой `margin`, `P`/`K`, `lr` сработали; batch-hard почти всегда
> обучается заметно лучше, чем случайные тройки.